In [28]:
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import GradientBoostingRegressor

sns.set_theme(style="whitegrid")

In [ ]:
Беру

In [29]:
df = pd.read_csv("housing.csv")
print(df.columns.tolist())

['longitude', 'latitude', 'housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income', 'median_house_value', 'ocean_proximity']


In [30]:

y = df["median_house_value"]
X = df.drop(columns=["median_house_value"])

# ручное кодирование ocean_proximity в числа
mapping = {v: i for i, v in enumerate(df["ocean_proximity"].astype(str).unique())}
X["ocean_proximity"] = X["ocean_proximity"].astype(str).map(mapping)

# теперь всё числовое
X = X.fillna(X.median(numeric_only=True))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [31]:
class MyBoost:
    def __init__(self, n=400, lr=0.05, depth=7, seed=42,
                 subsample=1.0, colsample_bytree=1.0):
        self.n = n
        self.lr = lr
        self.depth = depth
        self.seed = seed
        self.subsample = subsample
        self.colsample_bytree = colsample_bytree

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)

        n_samples, n_features = X.shape
        rng = np.random.default_rng(self.seed)

        self.initial_leaf = y.mean()
        preds = np.full(n_samples, self.initial_leaf)
        self.trees_ = []
        self.cols_ = []

        for _ in range(self.n):
            antigrad = y - preds

            m = max(1, int(self.subsample * n_samples))
            k = max(1, int(self.colsample_bytree * n_features))
            idx = rng.choice(n_samples, m, replace=False)
            cols = np.sort(rng.choice(n_features, k, replace=False))

            tree = DecisionTreeRegressor(
                max_depth=self.depth,
                random_state=self.seed,
                criterion="friedman_mse"
            )
            tree.fit(X[idx][:, cols], antigrad[idx])

            self.trees_.append(tree)
            self.cols_.append(cols)
            preds += self.lr * tree.predict(X[:, cols])

        fi = np.zeros(n_features)
        for t, cols in zip(self.trees_, self.cols_):
            imp = np.zeros(n_features)
            imp[cols] = t.feature_importances_
            fi += self.lr * imp
        s = fi.sum()
        self.feature_importances_ = fi / s if s > 0 else fi

        self.n_features_in_ = n_features
        return self

    def predict(self, X):
        X = np.asarray(X, float)
        preds = np.full(X.shape[0], self.initial_leaf)
        for t, cols in zip(self.trees_, self.cols_):
            preds += self.lr * t.predict(X[:, cols])
        return preds

In [32]:
results = []

gbm = GradientBoostingRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=3,
    subsample=0.8,
    random_state=42
)

start_time = time.time()
gbm.fit(X_train, y_train)
train_time = time.time() - start_time

preds = gbm.predict(X_test)

results.append({
    "Модель": "GradientBoostingRegressor",
    "Время обучения": round(train_time, 3),
    "MAE": round(mean_absolute_error(y_test, preds), 4),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, preds)), 4),
    "R2 Score": round(r2_score(y_test, preds), 4)
})

my = MyBoost(
    n=300,
    lr=0.05,
    depth=3,
    seed=42,
    subsample=0.8,
    colsample_bytree=0.8
)

start_time = time.time()
my.fit(X_train, y_train)
train_time = time.time() - start_time

preds = my.predict(X_test)

results.append({
    "Модель": "MyBoost",
    "Время обучения": round(train_time, 3),
    "MAE": round(mean_absolute_error(y_test, preds), 4),
    "RMSE": round(np.sqrt(mean_squared_error(y_test, preds)), 4),
    "R2 Score": round(r2_score(y_test, preds), 4)
})

df_results = pd.DataFrame(results)
display(df_results)

,Модель,Время обучения,MAE,RMSE,R2 Score
0,GradientBoostingRegressor,4.339,37118.0530,54167.0810,0.7761
1,MyBoost,3.473,36744.0222,53773.8059,0.7793


In [33]:
fi = pd.DataFrame({
    "feature": X.columns,
    "importance": my.feature_importances_
}).sort_values("importance", ascending=False)

fi.head(15)

,feature,importance
1,latitude,0.195030
0,longitude,0.187397
7,median_income,0.179578
5,population,0.126486
4,total_bedrooms,0.079815
2,housing_median_age,0.065871
8,ocean_proximity,0.064795
6,households,0.058318
3,total_rooms,0.042710


Вывод: градиентный бустинг позволяет 